# Phase 1 · Taxonomy and aliases

Phase 1 builds the complete master taxonomy for bacteria, fungi, and viruses, regardless of the current `run.organism_groups` selection. This lets later runs switch domains without rebuilding a partial taxonomy. Inputs are local source snapshots and the V2 migration copy; this phase makes no remote API calls and never writes to V2.

In [ ]:
from pathlib import Path
import inspect
import pandas as pd
from IPython.display import display
from pathogen_atlas_v3.common.io import load_yaml
from pathogen_atlas_v3.pipelines.phase1_taxonomy import run_phase1

CONFIG_PATH = Path("configs/taxonomy/source_files.yaml")
SETTINGS = load_yaml(CONFIG_PATH)
DEFAULT_CONFIG = load_yaml("configs/default.yaml")
print("Active Phase 2–5 selection:", DEFAULT_CONFIG["run"]["organism_groups"])
print("Phase 1 output:", DEFAULT_CONFIG["paths"]["taxonomy_release"])
source_rows=[]
for group, source in SETTINGS["sources"].items():
    for role in ("source", "variant_metadata"):
        item = source if role == "source" else source.get("variant_metadata")
        if item:
            p=Path(item["path"])
            source_rows.append({"group":group,"role":role,"source":item.get("source", source.get("source")),"release":item.get("release"),"path":str(p),"exists":p.exists()})
display(pd.DataFrame(source_rows))
print("\nImplementation used by this notebook:\n")
print(inspect.getsource(run_phase1))

## Run taxonomy adapters

The run writes normalized species, aliases, and virus variant tables. Re-running rebuilds these deterministic tables from the pinned local source files.

In [ ]:
RUN_PHASE = True
if RUN_PHASE:
    TAXONOMY_COUNTS = run_phase1(CONFIG_PATH, migration_source=DEFAULT_CONFIG["paths"]["migration_source"])
else:
    TAXONOMY_COUNTS = {"status":"not_run", "reason":"Set RUN_PHASE=True to rebuild taxonomy tables."}
TAXONOMY_COUNTS